In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

In [2]:
from src.backend import is_cuda_available

device = "cuda" if is_cuda_available() else "cpu"

device

'cuda'

In [3]:
numerical_features = ["age", "study_hours", "class_attendance", "sleep_hours"]
categorical_features = [
    "gender",
    "course",
    "internet_access",
    "sleep_quality",
    "study_method",
    "facility_rating",
    "exam_difficulty",
]

In [4]:
import pandas as pd
import numpy as np


def create_features(df: pd.DataFrame):
    X = df.copy()

    for p in [12, 14, 20]:
        X[f"_study_hours_sin_{p}"] = np.sin(2 * np.pi * X["study_hours"] / p).astype(
            "float32"
        )
        X[f"_class_attendance_sin_{p}"] = np.sin(
            2 * np.pi * X["class_attendance"] / p
        ).astype("float32")
        X[f"_study_hours_cos_{p}"] = np.cos(2 * np.pi * X["study_hours"] / p).astype(
            "float32"
        )

    for c in numerical_features:
        X[f"{c}_sq"] = X[c] ** 2

    X["feature_formula"] = (
        5.9051154511950499 * X["study_hours"]
        + 0.34540967058057986 * X["class_attendance"]
        + 1.423461171860262 * X["sleep_hours"]
        + 4.7819
    )

    X["_feature_formula"] = (
        6 * X["study_hours"]
        + 0.35 * X["class_attendance"]
        + 1.5 * X["sleep_hours"]
        + 5 * (X["sleep_quality"] == "good")
        + -5 * (X["sleep_quality"] == "poor")
        + 10 * (X["study_method"] == "coaching")
        + 5 * (X["study_method"] == "mixed")
        + 2 * (X["study_method"] == "group study")
        + 1 * (X["study_method"] == "online videos")
        + 4 * (X["facility_rating"] == "high")
        + -4 * (X["facility_rating"] == "low")
    )

    for c in ["age"]:
        for k in range(-1, 1):
            n = f"{c}_d{k}"
            X[n] = ((X[c] * 10**k) % 10).fillna(-1).astype("int8")

    for c in ["study_hours"]:
        for k in range(0, 3):
            n = f"{c}_d{k}"
            X[n] = ((X[c] * 10**k) % 10).fillna(-1).astype("int8")

    for c in ["class_attendance"]:
        for k in range(-1, 2):
            n = f"{c}_d{k}"
            X[n] = ((X[c] * 10**k) % 10).fillna(-1).astype("int8")

    for c in ["sleep_hours"]:
        for k in range(0, 2):
            n = f"{c}_d{k}"
            X[n] = ((X[c] * 10**k) % 10).fillna(-1).astype("int8")

    return X

In [5]:
train = pd.read_csv("../train.csv")
test = pd.read_csv("../test.csv")

train.drop("id", axis=1, inplace=True)

In [6]:
train_fe = create_features(train)
test_fe = create_features(test)

In [7]:
y_train = train_fe["exam_score"]
X_train: pd.DataFrame = train_fe.drop("exam_score", axis=1)
X_test: pd.DataFrame = test_fe

In [8]:
X_train.head()

,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,...,age_d-1,age_d0,study_hours_d0,study_hours_d1,study_hours_d2,class_attendance_d-1,class_attendance_d0,class_attendance_d1,sleep_hours_d0,sleep_hours_d1
0,21,female,b.sc,7.91,98.8,no,4.9,average,online videos,low,...,2,1,7,9,1,9,8,8,4,9
1,18,other,diploma,4.95,94.8,yes,4.7,poor,self-study,medium,...,1,8,4,9,5,9,4,8,4,7
2,20,female,b.sc,4.68,92.6,yes,5.8,poor,coaching,high,...,2,0,4,6,8,9,2,6,5,8
3,19,male,b.sc,2.00,49.5,yes,8.3,average,group study,high,...,1,9,2,0,0,4,9,5,8,3
4,23,male,bca,7.65,86.9,yes,9.6,good,self-study,high,...,2,3,7,6,5,8,6,9,9,6


In [9]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
from catboost import CatBoostRegressor


def catboost_fit(
    model: CatBoostRegressor,
    X_train: pd.DataFrame,
    y_train: pd.DataFrame,
    X_valid: pd.DataFrame,
    y_valid: pd.DataFrame,
):
    model.set_params(early_stopping_rounds=50)

    model.fit(
        X_train,
        y_train,
        eval_set=(X_valid, y_valid),
        verbose=False,
        cat_features=categorical_features,
    )

    return model


def catboost_predict(model: CatBoostRegressor, X_valid: pd.DataFrame):
    return model.predict(X_valid)

In [14]:
from src.predict import ensemble_predict

train, oof_pred_df, test_pred_df = ensemble_predict(
    {
        "CatBoost": (
            CatBoostRegressor(
                iterations=2000,
                depth=6,
                loss_function="RMSE",
                eval_metric="RMSE",
                random_seed=42,
                thread_count=-1,
                verbose=False,
                allow_writing_files=False,
                task_type="GPU" if device == "cuda" else "CPU",
            ),
            catboost_fit,
            catboost_predict,
            False,
            True,
        ),
    },
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    kf=kf,
    numerical_features=[],
    checkpoint_dir="../speculative",
    recompute=False,
    ignore=[],
)

[checkpoint] CatBoost: loaded


In [15]:
submission = X_test[["id"]]
submission["exam_score"] = test_pred_df["CatBoost"]
submission.to_csv("../speculative.csv", index=False)